# CardioVLM GPU Smoke Test

Validates the full CardioVLM stack on an A100 GPU:
- Qwen2.5-VL-7B model loading
- CineMemModel with dual memory initialization
- Forward pass and generation
- NGRPO advantage calibration

**Runtime:** A100 (80 GB) recommended. Should fit on 40 GB with headroom.

In [ ]:
# === Cell 1: Colab Setup ===
# Uncomment these lines when running in Colab:
# !pip install git+https://github.com/<your-username>/CardioVLM.git
# !pip install torch transformers accelerate peft safetensors nibabel qwen-vl-utils

In [ ]:
# === Cell 2: Verify GPU ===
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — switch to A100 runtime")

In [ ]:
# === Cell 3: Load Qwen2.5-VL base model ===
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Loading processor...")
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
print("Loading model (this takes 2-3 min on A100)...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
print(f"Model loaded. Parameters: {sum(p.numel() for p in base_model.parameters()) / 1e9:.1f}B")
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# === Cell 4: Initialize CineMemModel with dual memory ===
from cardio.vlm.config import build_cinemem_config
from cardio.vlm.model import CineMemModel
from cardio.vlm.constants import ALL_SPECIAL_TOKENS

# Add special tokens
tokenizer.add_special_tokens({"additional_special_tokens": ALL_SPECIAL_TOKENS})
base_model.resize_token_embeddings(len(tokenizer))

config = build_cinemem_config({
    "cinemem": {
        "query_len": 8,
        "short_mem_len": 8,
        "long_mem_len": 16,
        "use_dual_memory": True,
        "former_backend": "tiny_transformer",
        "fix_mrope_temporal": True,
    }
})

cinemem = CineMemModel(base_model, tokenizer, processor, config)
print("CineMemModel initialized with dual memory")
print(f"Total memory after init: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# === Cell 5: Test forward pass with dummy input ===
from PIL import Image
import numpy as np

# Create a dummy cardiac MRI-like image (grayscale, 256x256)
dummy_image = Image.fromarray(
    np.random.randint(0, 255, (256, 256), dtype=np.uint8), mode="L"
).convert("RGB")

prompt_text = "Assess the left ventricular systolic function based on this cardiac MRI."

messages = [
    {"role": "user", "content": [
        {"type": "image", "image": dummy_image},
        {"type": "text", "text": prompt_text},
    ]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[dummy_image], return_tensors="pt", padding=True)
inputs = {k: v.to(base_model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

print("Running forward pass...")
with torch.no_grad():
    outputs = base_model(**inputs, output_hidden_states=True)
print(f"Forward pass OK. Logits shape: {outputs.logits.shape}")
print(f"Hidden states captured: {len(outputs.hidden_states)} layers")

In [ ]:
# === Cell 6: Test generation with memory invocation ===
# CineMemModel.generate takes (images, prompts) — it builds processor
# inputs internally via build_processor_inputs.

prompt_text = "Assess the left ventricular systolic function based on this cardiac MRI."

print("Running generation (max 100 tokens)...")
with torch.no_grad():
    generated = cinemem.generate(
        images=[dummy_image],
        prompts=[prompt_text],
        max_new_tokens=100,
        enable_cinemem=True,
        skip_special_tokens=False,
    )

# generate() returns list[str] — already decoded text
decoded = generated[0]
print(f"\nGenerated text:\n{decoded}")
print(f"\nInvocation log: {cinemem.invocation_log}")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
# === Cell 7: Test NGRPO with real rewards ===
from cardio.trainer.grpo import NGRPOTrainer
from cardio.trainer.rewards.composite import CompositeRewardEngine
from cardio.trainer.rewards.vprm import ACCAHAVerifier
from cardio.trainer.rewards.memory_penalty import MemoryInvocationVerifier
from cardio.trainer.rewards.dcr import DivideConquerEvaluator, AutoMetricConverter

# model=None is safe here — we only test advantage math, not loss_from_samples
trainer = NGRPOTrainer(model=None, r_max=1.0, epsilon_neg=0.16, epsilon_pos=0.24)

# Simulate a group of 4 rollouts with varied rewards
rewards = torch.tensor([0.3, -1.0, 0.7, 0.0])
advantages = trainer.compute_calibrated_advantages(rewards)
print(f"Rewards: {rewards}")
print(f"Calibrated advantages: {advantages}")
print(f"Best rollout advantage: {advantages.max():.4f} (should be positive)")
print(f"Worst rollout advantage: {advantages.min():.4f} (should be negative)")

In [ ]:
# === Cell 8: Memory usage summary ===
print("\n=== GPU SMOKE TEST COMPLETE ===")
peak_gb = torch.cuda.max_memory_allocated() / 1e9
current_gb = torch.cuda.memory_allocated() / 1e9
print(f"Peak VRAM used: {peak_gb:.1f} GB")
print(f"Current VRAM allocated: {current_gb:.1f} GB")
if peak_gb < 35:
    print("STATUS: Fits comfortably on A100-40GB. Ready for training.")
elif peak_gb < 40:
    print("STATUS: Tight on A100-40GB. Use gradient checkpointing for training.")
else:
    print("STATUS: May need A100-80GB or model parallelism for training.")